In [1]:
import json
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc

REPO_ROOT = Path("../..").resolve()

# ===== PARÁMETROS =====
NOMBRE = "test_27"                                    # nombre del JSON
VIDEO_IN = REPO_ROOT / "data" / "raw" / "test (27).mp4"
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
MARGEN_CAMPO = 5.0                                    # m fuera del campo que se toleran
# ======================

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)

frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(v) for k, v in data["color_equipo_bgr"].items()}

def en_campo(c):
    """Filtro de cordura: descarta proyecciones explosivas o muy fuera del campo.
    El 2% de posiciones del pipeline caen a miles de metros (homografías degeneradas con
    pocos keypoints); sin este filtro, esos outliers rompen la escala de la pizarra."""
    return (c is not None and -MARGEN_CAMPO <= c[0] <= 105 + MARGEN_CAMPO
            and -MARGEN_CAMPO <= c[1] <= 68 + MARGEN_CAMPO)

print(f"JSON cargado: {JSON_IN.name}")
print(f"Vídeo: {data['video']} | Frames: {N_FRAMES}")
print(f"Colores equipo (BGR): {color_equipo_bgr}")
print(f"Vídeo existe: {VIDEO_IN.exists()}")

JSON cargado: integrado_test_27.json
Vídeo: test (27).mp4 | Frames: 750
Colores equipo (BGR): {0: (254, 246, 252), 1: (110, 254, 250)}
Vídeo existe: True


In [2]:
def dibujar_campo(ax):
    """Campo de fútbol 2D a escala (105x68 m)."""
    ax.add_patch(Rectangle((0, 0), 105, 68, color="#3a7d34", zorder=0))
    lc, lw = "white", 2
    ax.add_patch(Rectangle((0, 0), 105, 68, fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([52.5, 52.5], [0, 68], color=lc, linewidth=lw)
    ax.add_patch(Circle((52.5, 34), 9.15, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((52.5, 34), 0.4, color=lc))
    ax.add_patch(Rectangle((0, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((11, 34), 0.4, color=lc))
    ax.add_patch(Arc((11, 34), 18.3, 18.3, theta1=-53, theta2=53, color=lc, linewidth=lw))
    ax.add_patch(Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((99.5, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((94, 34), 0.4, color=lc))
    ax.add_patch(Arc((94, 34), 18.3, 18.3, theta1=127, theta2=233, color=lc, linewidth=lw))
    ax.set_xlim(-5, 110); ax.set_ylim(-5, 73)
    ax.set_aspect("equal"); ax.axis("off")

print("Función dibujar_campo definida")

Función dibujar_campo definida


In [3]:
# ---- Trayectorias por identidad: filtrar saltos imposibles, suavizar, rellenar huecos ----
VENTANA = 5         # media móvil más corta: con los saltos ya filtrados no hace falta más
MAX_HUECO = 15      # huecos más cortos se interpolan
MAX_VEL = 1.2       # m/frame: velocidad máxima física de un jugador (~30 m/s, generoso para
                    # absorber carreras y pequeños errores; por encima = salto de homografía)

# Agrupar posiciones SANAS por (id, equipo, rol)
tray = defaultdict(dict)
meta = {}
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") not in ("jugador", "portero"):
            continue
        if not en_campo(d.get("campo")) or d.get("equipo") not in (0, 1):
            continue
        key = (d["id"], d["equipo"], d["rol"])
        tray[key][f_idx] = tuple(d["campo"])
        meta[key] = {"equipo": d["equipo"], "rol": d["rol"]}

def filtrar_saltos(serie):
    """Elimina posiciones que implican velocidad físicamente imposible respecto a la anterior
    aceptada. Esos saltos son ruido de homografía (cambio brusco de calibración), no movimiento."""
    if len(serie) < 2:
        return dict(serie)
    fs = sorted(serie)
    limpia = {fs[0]: serie[fs[0]]}
    f_ant, p_ant = fs[0], serie[fs[0]]
    for f in fs[1:]:
        p = serie[f]
        dt = f - f_ant
        vel = np.hypot(p[0]-p_ant[0], p[1]-p_ant[1]) / max(dt, 1)
        if vel <= MAX_VEL:
            limpia[f] = p
            f_ant, p_ant = f, p
        # si el salto es imposible, se descarta este punto y se mantiene el anterior como referencia
    return limpia

def suavizar_y_rellenar(serie):
    serie = filtrar_saltos(serie)        # NUEVO: filtrar saltos antes de nada
    if not serie:
        return {}
    fs = sorted(serie)
    completa = {}
    for a, b in zip(fs, fs[1:]):
        completa[a] = serie[a]
        hueco = b - a
        if 1 < hueco <= MAX_HUECO:
            xa, ya = serie[a]; xb, yb = serie[b]
            for k in range(1, hueco):
                t = k / hueco
                completa[a + k] = (xa + t*(xb-xa), ya + t*(yb-ya))
    completa[fs[-1]] = serie[fs[-1]]
    fs2 = sorted(completa)
    arr = np.array([completa[f] for f in fs2])
    suav, h = {}, VENTANA // 2
    for i, f in enumerate(fs2):
        lo, hi = max(0, i-h), min(len(arr), i+h+1)
        suav[f] = tuple(arr[lo:hi].mean(axis=0))
    return suav

tray_suave = {key: suavizar_y_rellenar(serie) for key, serie in tray.items()}

# Árbitros aparte
arb_tray = defaultdict(dict)
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") == "arbitro" and en_campo(d.get("campo")):
            arb_tray[d["id"]][f_idx] = tuple(d["campo"])
arb_suave = {aid: suavizar_y_rellenar(s) for aid, s in arb_tray.items()}

# Balón aparte
balon_tray = {}
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") == "balon" and en_campo(d.get("campo")):
            balon_tray[f_idx] = tuple(d["campo"])

# Reindexar por frame
pos_por_frame = defaultdict(list)
for key, serie in tray_suave.items():
    eq, rol = meta[key]["equipo"], meta[key]["rol"]
    for f, (x, y) in serie.items():
        pos_por_frame[f].append((x, y, eq, rol))
arb_por_frame = defaultdict(list)
for aid, serie in arb_suave.items():
    for f, (x, y) in serie.items():
        arb_por_frame[f].append((x, y))

n_ids = len(tray_suave)
n_raw = sum(len(s) for s in tray.values())
n_suave = sum(len(s) for s in tray_suave.values())
print(f"Identidades de campo: {n_ids}")
print(f"Posiciones: {n_raw} brutas -> {n_suave} tras filtrar saltos + rellenar huecos")
print(f"Árbitros: {len(arb_suave)} | Balón: {len(balon_tray)} frames")

Identidades de campo: 42
Posiciones: 14329 brutas -> 14094 tras filtrar saltos + rellenar huecos
Árbitros: 26 | Balón: 0 frames


In [4]:
# ---- Pizarra 2D animada: equipos por color, porteros cuadrados, árbitros negros ----
cap = cv2.VideoCapture(str(VIDEO_IN))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

# BGR del JSON -> RGB normalizado para matplotlib
col_eq = {eq: (np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}

OUTPUT_VIDEO = REPO_ROOT / "outputs" / f"pizarra_fase5_{NOMBRE}.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)
PIZARRA_W, PIZARRA_H = 1100, 800
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
                         fps, (PIZARRA_W, PIZARRA_H))

for f in range(N_FRAMES):
    fig, ax = plt.subplots(figsize=(11, 8), dpi=100)
    dibujar_campo(ax)

    for (x, y, eq, rol) in pos_por_frame.get(f, []):
        c = col_eq.get(eq, (0.5, 0.5, 0.5))
        marca = "s" if rol == "portero" else "o"
        ax.scatter(x, y, marker=marca, c=[c], s=160, edgecolors="white",
                   linewidth=1.5, zorder=5)

    for (x, y) in arb_por_frame.get(f, []):
        ax.scatter(x, y, marker="^", c="black", s=130, edgecolors="white",
                   linewidth=1.2, zorder=5)

    if f in balon_tray:
        bx, by = balon_tray[f]
        ax.scatter(bx, by, c="yellow", s=90, edgecolors="black", linewidth=1.5, zorder=6)

    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    img = cv2.resize(img, (PIZARRA_W, PIZARRA_H))
    writer.write(img)
    plt.close(fig)

    if f % 50 == 0:
        print(f"Pizarra: {f}/{N_FRAMES} frames...")

writer.release()
print(f"\nPizarra guardada en: {OUTPUT_VIDEO}")

Pizarra: 0/750 frames...
Pizarra: 50/750 frames...
Pizarra: 100/750 frames...
Pizarra: 150/750 frames...
Pizarra: 200/750 frames...
Pizarra: 250/750 frames...
Pizarra: 300/750 frames...
Pizarra: 350/750 frames...
Pizarra: 400/750 frames...
Pizarra: 450/750 frames...
Pizarra: 500/750 frames...
Pizarra: 550/750 frames...
Pizarra: 600/750 frames...
Pizarra: 650/750 frames...
Pizarra: 700/750 frames...

Pizarra guardada en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\pizarra_fase5_test_27.mp4
